# PyTorch Mini Projects

**PyTorch Interview Preparation · Notebook 12 · Estimated study time: 5–8 hours**

This is an exercise-first capstone. Each project follows **problem → data → Dataset/DataLoader → model → loss → optimizer → training → validation → debugging → evaluation → explanation**. Attempt every starter cell before revealing its solution. Reference implementation cells are tagged `solution` and collapsed in supporting frontends; expand them when you want the complete executable path.

All data is synthetic, learnable, deterministic, and offline. Every project runs on CPU.


## 1. How to Approach a PyTorch Project

### 🔥 Interview Essential

1. Understand the task.
2. Inspect input and target shapes.
3. Choose the output representation.
4. Choose a matching loss.
5. Build `Dataset` / `DataLoader` objects.
6. Build a reasonable baseline model.
7. Select an optimizer.
8. Train.
9. Validate.
10. Debug shapes, dtypes, devices, gradients, and updates.
11. Evaluate with a task-appropriate metric.
12. Explain every design decision.

Interviewers usually care more about this reasoning chain than perfect API recall. State assumptions, start simple, verify one batch, and only then train.


## 2. Reusable Setup


In [ ]:
import math
import random

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split

torch.manual_seed(42)
random.seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Using:", device)


In [ ]:
def pending(name, *values):
    # Keep starter cells executable without silently accepting missing work.
    if any(value is None for value in values):
        print(f"PENDING: {name} — attempt the starter, then reveal the solution")
        return True
    return False

def split_dataset(dataset, train_fraction=0.8, seed=42):
    n_train = int(len(dataset) * train_fraction)
    return random_split(
        dataset,
        [n_train, len(dataset) - n_train],
        generator=torch.Generator().manual_seed(seed),
    )

def parameter_snapshot(model):
    return [p.detach().clone() for p in model.parameters()]

def parameters_changed(before, model):
    return any(not torch.equal(old, new.detach().cpu())
               for old, new in zip(before, model.cpu().parameters()))

print("Reusable helpers ready.")


### Utility design exercise

Implement `train_one_epoch(model, loader, loss_fn, optimizer, device)` and `evaluate_classifier(model, loader, loss_fn, device)`. Keep them readable. What must differ for regression, binary classification, multiclass classification, and dictionary batches? Avoid an abstraction so generic that it hides logits, targets, or metrics.


In [ ]:
# Your turn: write signatures and pseudocode first.
train_one_epoch_draft = None
evaluate_classifier_draft = None
pending("reusable utility drafts", train_one_epoch_draft, evaluate_classifier_draft)


<details><summary>Design note</summary>

A small amount of duplication is healthy here. Each project below uses a short task-specific loop so output/target transformations remain visible.

</details>


## 3. Project 1 — Linear Regression

### Problem

Recover a noisy relationship: `y = 3x₁ - 2x₂ + 1.5 + noise`. One sample has two float features and one continuous float target.


In [ ]:
torch.manual_seed(101)
num_samples = 1000
X_reg = torch.randn(num_samples, 2)
noise = 0.2 * torch.randn(num_samples, 1)
y_reg = 3 * X_reg[:, 0:1] - 2 * X_reg[:, 1:2] + 1.5 + noise
assert X_reg.shape == (1000, 2) and y_reg.shape == (1000, 1)
print(X_reg.shape, y_reg.shape, X_reg.dtype, y_reg.dtype)


### Before coding

1. Regression or classification?
2. What output shape should a batch have?
3. Which loss and metric fit?
4. Should the last layer have an activation?
5. Why would accuracy be meaningless?


<details><summary>Reveal reasoning</summary>

Regression; output `(B, 1)`; `MSELoss` with MAE/MSE/RMSE metrics; no classification activation. Exact equality accuracy is unstable and does not express error magnitude.

</details>


### Data pipeline — your turn


In [ ]:
# Build TensorDataset, deterministic 80/20 split, and loaders.
# Required: train batch 32/shuffle; validation batch 64/no shuffle.
reg_train_loader = None
reg_val_loader = None
if not pending("regression loaders", reg_train_loader, reg_val_loader):
    xb, yb = next(iter(reg_train_loader))
    assert xb.shape[1:] == (2,) and yb.shape[1:] == (1,)


In [ ]:
reg_dataset = TensorDataset(X_reg, y_reg)
reg_train_ds, reg_val_ds = split_dataset(reg_dataset)
reg_train_loader = DataLoader(reg_train_ds, batch_size=32, shuffle=True)
reg_val_loader = DataLoader(reg_val_ds, batch_size=64, shuffle=False)
assert len(reg_train_ds) == 800 and len(reg_val_ds) == 200
print("✅ regression loaders:", next(iter(reg_train_loader))[0].shape)


### Model, loss, and optimizer — your turn


In [ ]:
# Required shape path: 2 → 16 → 8 → 1. Do not add sigmoid/softmax.
reg_model_draft = None
reg_loss_draft = None
reg_optimizer_draft = None
pending("regression model/loss/optimizer", reg_model_draft, reg_loss_draft, reg_optimizer_draft)


In [ ]:
reg_model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 8), nn.ReLU(),
    nn.Linear(8, 1),
).to(device)
reg_loss_fn = nn.MSELoss()
reg_optimizer = optim.Adam(reg_model.parameters(), lr=0.02)
test_x = torch.randn(8, 2, device=device)
assert reg_model(test_x).shape == (8, 1)
assert isinstance(reg_loss_fn, nn.MSELoss)
print("✅ regression model output:", reg_model(test_x).shape)


### Training — your turn

Write one complete epoch: `train()`, move both tensors, zero gradients, forward, loss, backward, step. Then validate under `eval()` and `inference_mode()`. Store both losses. Verify the first batch has finite loss, non-`None` gradients, and that at least one parameter changes.


In [ ]:
reg_history_draft = None
pending("regression training loop", reg_history_draft)


In [ ]:
reg_history = {"train": [], "val": []}
reg_before = [p.detach().cpu().clone() for p in reg_model.parameters()]
for epoch in range(18):
    reg_model.train()
    train_sum = 0.0
    for xb, yb in reg_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        reg_optimizer.zero_grad()
        pred = reg_model(xb)
        loss = reg_loss_fn(pred, yb)
        assert torch.isfinite(loss)
        loss.backward()
        assert all(p.grad is not None for p in reg_model.parameters() if p.requires_grad)
        reg_optimizer.step()
        train_sum += loss.item() * xb.size(0)
    reg_model.eval()
    val_sum = 0.0
    with torch.inference_mode():
        for xb, yb in reg_val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_sum += reg_loss_fn(reg_model(xb), yb).item() * xb.size(0)
    reg_history["train"].append(train_sum / len(reg_train_ds))
    reg_history["val"].append(val_sum / len(reg_val_ds))
reg_after = [p.detach().cpu().clone() for p in reg_model.parameters()]
assert any(not torch.equal(a, b) for a, b in zip(reg_before, reg_after))
print(f"train={reg_history['train'][-1]:.4f} val={reg_history['val'][-1]:.4f}")


In [ ]:
reg_model.eval()
reg_preds, reg_targets = [], []
with torch.inference_mode():
    for xb, yb in reg_val_loader:
        reg_preds.append(reg_model(xb.to(device)).cpu())
        reg_targets.append(yb)
reg_preds, reg_targets = torch.cat(reg_preds), torch.cat(reg_targets)
reg_mse = ((reg_preds - reg_targets) ** 2).mean().item()
reg_mae = (reg_preds - reg_targets).abs().mean().item()
reg_rmse = reg_mse ** 0.5
assert reg_preds.shape == reg_targets.shape == (200, 1)
print(f"MSE={reg_mse:.4f} MAE={reg_mae:.4f} RMSE={reg_rmse:.4f}")


### Interview discussion

- Why MSE? When might `L1Loss` be preferable?
- Why no softmax? What output shape is required?
- What happens when predictions are `(B,1)` but targets are `(B,)`? Why is unintended broadcasting dangerous?
- **What would you try next?** Check a linear baseline, normalize differently scaled features, inspect residuals, and tune learning rate based on evidence—not reflexively.


<details><summary>Concise answers</summary>

MSE strongly penalizes large errors and matches Gaussian-noise assumptions. L1 is more robust to outliers. Softmax represents class probabilities, not an unbounded continuous target. `(B,1) - (B,)` can broadcast to `(B,B)`, silently optimizing the wrong pairwise errors.

</details>


### Before moving on

- [ ] I understand input and target shapes.
- [ ] I chose the loss and built the model myself.
- [ ] I wrote training and validation loops and calculated the metric.
- [ ] I can explain the architecture and why the loss is correct.
- [ ] I can describe at least two likely bugs and one sensible improvement.


## 4. Project 2 — Binary Classification

### 🔥 Interview Essential

Return one **raw logit** per sample, use float targets shaped `(B,1)`, and train with `BCEWithLogitsLoss`. Apply sigmoid only for inference.


In [ ]:
torch.manual_seed(202)
X_bin = torch.randn(1500, 10)
true_w_bin = torch.randn(10, 1)
scores_bin = X_bin @ true_w_bin + 0.15 * torch.randn(1500, 1)
y_bin = (scores_bin > 0).float()
assert X_bin.shape == (1500, 10) and y_bin.shape == (1500, 1)
print("positive rate:", y_bin.mean().item())


### Plan before coding

What should the model output? Target dtype? Loss? Should sigmoid be inside `forward()`? How are logits converted into predictions?


<details><summary>Reveal reasoning</summary>

Output `(B,1)` raw logits; float target `(B,1)`; `BCEWithLogitsLoss`; no sigmoid in the model; at inference use `sigmoid(logits) >= threshold`.

</details>


In [ ]:
# Your turn: loaders and a 10 → 32 → 16 → 1 classifier.
bin_model_draft = None
bin_loss_draft = None
pending("binary pipeline and model", bin_model_draft, bin_loss_draft)


In [ ]:
bin_train_ds, bin_val_ds = split_dataset(TensorDataset(X_bin, y_bin))
bin_train_loader = DataLoader(bin_train_ds, batch_size=32, shuffle=True)
bin_val_loader = DataLoader(bin_val_ds, batch_size=64, shuffle=False)
bin_model = nn.Sequential(
    nn.Linear(10, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1),
).to(device)
bin_loss_fn = nn.BCEWithLogitsLoss()
bin_optimizer = optim.AdamW(bin_model.parameters(), lr=0.01)
assert bin_model(torch.randn(8, 10, device=device)).shape == (8, 1)
assert not any(isinstance(m, nn.Sigmoid) for m in bin_model.modules())
print("✅ binary pipeline ready")


In [ ]:
# Your turn: write training + validation, then compute accuracy.
bin_history_draft = None
pending("binary training", bin_history_draft)


In [ ]:
def run_binary_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = correct = total = 0
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if training:
                optimizer.zero_grad()
            logits = model(xb)
            assert logits.shape == yb.shape and yb.dtype == torch.float32
            loss = loss_fn(logits, yb)
            if training:
                loss.backward(); optimizer.step()
            total_loss += loss.item() * xb.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5).float() == yb).sum().item()
            total += yb.numel()
    return total_loss / len(loader.dataset), correct / total

bin_history = {"train_loss": [], "val_loss": [], "val_acc": []}
for epoch in range(12):
    tr_loss, _ = run_binary_epoch(bin_model, bin_train_loader, bin_loss_fn, bin_optimizer)
    va_loss, va_acc = run_binary_epoch(bin_model, bin_val_loader, bin_loss_fn)
    bin_history["train_loss"].append(tr_loss)
    bin_history["val_loss"].append(va_loss)
    bin_history["val_acc"].append(va_acc)
assert 0 <= va_acc <= 1
print(f"val loss={va_loss:.4f} accuracy={va_acc:.3f}")


In [ ]:
bin_model.eval()
all_logits, all_y = [], []
with torch.inference_mode():
    for xb, yb in bin_val_loader:
        all_logits.append(bin_model(xb.to(device)).cpu()); all_y.append(yb)
all_logits, all_y = torch.cat(all_logits), torch.cat(all_y)

def binary_metrics(logits, targets, threshold=0.5):
    pred = (torch.sigmoid(logits) >= threshold)
    true = targets.bool()
    tp = (pred & true).sum().item(); fp = (pred & ~true).sum().item()
    tn = (~pred & ~true).sum().item(); fn = (~pred & true).sum().item()
    accuracy = (tp + tn) / max(tp + fp + tn + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "accuracy": accuracy, "precision": precision, "recall": recall}

for threshold in (0.5, 0.7):
    result = binary_metrics(all_logits, all_y, threshold)
    assert all(0 <= result[k] <= 1 for k in ("accuracy", "precision", "recall"))
    print(threshold, result)


### Threshold exercise

Explain the changed confusion counts at `0.5` and `0.7`. A higher threshold usually produces fewer positive predictions; precision may rise while recall may fall. This is a general trade-off, not a guaranteed monotonic result for every finite dataset.

**What would you try next?** Choose the threshold against the business cost, inspect class balance, add class weighting only if justified, and calibrate probabilities if decisions depend on them.


### Before moving on

- [ ] I understand input and target shapes.
- [ ] I chose the loss and built the model myself.
- [ ] I wrote training and validation loops and calculated the metric.
- [ ] I can explain the architecture and why the loss is correct.
- [ ] I can describe at least two likely bugs and one sensible improvement.


## 5. Project 3 — Multiclass Classification

Predict exactly one of five classes from 20 numeric features.


In [ ]:
torch.manual_seed(303)
num_samples_mc, num_features_mc, num_classes_mc = 2000, 20, 5
X_mc = torch.randn(num_samples_mc, num_features_mc)
true_w_mc = torch.randn(num_features_mc, num_classes_mc)
scores_mc = X_mc @ true_w_mc + 0.2 * torch.randn(num_samples_mc, num_classes_mc)
y_mc = scores_mc.argmax(dim=1)
assert X_mc.shape == (2000, 20) and y_mc.shape == (2000,)
assert y_mc.dtype == torch.long
print(torch.bincount(y_mc))


### Planning questions

Output shape? Loss? Target shape/dtype? Softmax before loss? Prediction operation?


<details><summary>Reveal reasoning</summary>

`(B,5)` logits; `CrossEntropyLoss`; `(B,)` long class indices; no softmax before loss; `argmax(dim=1)` for top-1 predictions.

</details>


In [ ]:
# Your turn: loaders and 20 → 64 → Dropout(0.2) → 32 → 5.
mc_model_draft = None
mc_loss_draft = None
pending("multiclass pipeline", mc_model_draft, mc_loss_draft)


In [ ]:
mc_train_ds, mc_val_ds = split_dataset(TensorDataset(X_mc, y_mc))
mc_train_loader = DataLoader(mc_train_ds, batch_size=32, shuffle=True)
mc_val_loader = DataLoader(mc_val_ds, batch_size=64, shuffle=False)
mc_model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 5),
).to(device)
mc_loss_fn = nn.CrossEntropyLoss()
mc_optimizer = optim.AdamW(mc_model.parameters(), lr=0.01)
assert mc_model(torch.randn(7, 20, device=device)).shape == (7, 5)
print("✅ multiclass model ready")


In [ ]:
# Your turn: implement train_one_epoch and evaluate.
mc_train_one_epoch_draft = None
mc_evaluate_draft = None
pending("multiclass loops", mc_train_one_epoch_draft, mc_evaluate_draft)


In [ ]:
def run_multiclass_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = correct = total = 0
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if training: optimizer.zero_grad()
            logits = model(xb)
            assert logits.ndim == 2 and yb.dtype == torch.long
            loss = loss_fn(logits, yb)
            if training: loss.backward(); optimizer.step()
            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item(); total += yb.numel()
    return total_loss / len(loader.dataset), correct / total

mc_history = {"train_loss": [], "val_loss": [], "val_acc": []}
for epoch in range(14):
    tr_loss, _ = run_multiclass_epoch(mc_model, mc_train_loader, mc_loss_fn, mc_optimizer)
    va_loss, va_acc = run_multiclass_epoch(mc_model, mc_val_loader, mc_loss_fn)
    mc_history["train_loss"].append(tr_loss); mc_history["val_loss"].append(va_loss)
    mc_history["val_acc"].append(va_acc)
print(f"train={tr_loss:.4f} val={va_loss:.4f} accuracy={va_acc:.3f}")
print("Overfitting check: compare the final train/validation trend, not one epoch.")


In [ ]:
mc_model.eval(); mc_logits, mc_targets = [], []
with torch.inference_mode():
    for xb, yb in mc_val_loader:
        mc_logits.append(mc_model(xb.to(device)).cpu()); mc_targets.append(yb)
mc_logits, mc_targets = torch.cat(mc_logits), torch.cat(mc_targets)
mc_pred = mc_logits.argmax(1)
confusion = torch.zeros(num_classes_mc, num_classes_mc, dtype=torch.int64)
for actual, predicted in zip(mc_targets, mc_pred):
    confusion[actual, predicted] += 1  # rows actual, columns predicted
top3 = mc_logits.topk(k=3, dim=1).indices
top3_accuracy = (top3 == mc_targets.unsqueeze(1)).any(dim=1).float().mean().item()
assert confusion.sum() == len(mc_val_ds) and 0 <= top3_accuracy <= 1
print("Confusion matrix (actual rows, predicted columns):\n", confusion)
print(f"top-3 accuracy={top3_accuracy:.3f}")


Top-k accuracy is useful when many classes are plausible and downstream review can inspect several candidates. It does not replace top-1 or per-class analysis.

**What would you try next?** Inspect confused classes, compare a linear baseline, normalize features if needed, and add capacity or regularization only after reading train/validation curves.


### Before moving on

- [ ] I understand input and target shapes.
- [ ] I chose the loss and built the model myself.
- [ ] I wrote training and validation loops and calculated the metric.
- [ ] I can explain the architecture and why the loss is correct.
- [ ] I can describe at least two likely bugs and one sensible improvement.


## 6. Project 4 — CNN Image Classification

Classify noisy `1×16×16` images containing a bright vertical stripe, horizontal stripe, or diagonal. No downloads or `torchvision`.

### 🔥 Interview Essential

`Conv2d` expects `(B,C,H,W)`, here `(B,1,16,16)`.


In [ ]:
def make_pattern_image(label, size=16):
    image = 0.20 * torch.randn(1, size, size)
    if label == 0:
        col = random.randint(3, size - 4); image[:, :, col-1:col+2] += 1.5
    elif label == 1:
        row = random.randint(3, size - 4); image[:, row-1:row+2, :] += 1.5
    else:
        offset = random.randint(-2, 2)
        for r in range(size):
            c = r + offset
            if 0 <= c < size: image[:, r, max(0,c-1):min(size,c+2)] += 1.5
    return image

random.seed(404); torch.manual_seed(404)
cnn_labels = torch.arange(1500) % 3
cnn_images = torch.stack([make_pattern_image(int(label)) for label in cnn_labels])
perm = torch.randperm(1500); cnn_images, cnn_labels = cnn_images[perm], cnn_labels[perm]
assert cnn_images.shape == (1500, 1, 16, 16)
print(cnn_images.shape, torch.bincount(cnn_labels))


### Shape calculation — do this before running

Fill the path: `(B,1,16,16)` → Conv `1→8` → pool → Conv `8→16` → pool → flatten → Linear 32 → 3 logits.


<details><summary>Reveal shapes</summary>

`(B,1,16,16) → (B,8,16,16) → (B,8,8,8) → (B,16,8,8) → (B,16,4,4) → (B,256) → (B,32) → (B,3)`.

</details>


In [ ]:
# Your turn: build the CNN and calculate parameters for conv1, conv2, fc1.
cnn_model_draft = None
cnn_parameter_counts_draft = None
pending("CNN and parameter counts", cnn_model_draft, cnn_parameter_counts_draft)


In [ ]:
class PatternCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(16 * 4 * 4, 32)
        self.fc2 = nn.Linear(32, 3)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        return self.fc2(torch.relu(self.fc1(x)))

cnn_model = PatternCNN().to(device)
counts = {
    "conv1": 8 * 1 * 3 * 3 + 8,
    "conv2": 16 * 8 * 3 * 3 + 16,
    "fc1": (16 * 4 * 4) * 32 + 32,
}
assert cnn_model.conv1.weight.numel() + cnn_model.conv1.bias.numel() == counts["conv1"]
assert cnn_model.conv2.weight.numel() + cnn_model.conv2.bias.numel() == counts["conv2"]
assert cnn_model.fc1.weight.numel() + cnn_model.fc1.bias.numel() == counts["fc1"]
assert cnn_model(torch.randn(4, 1, 16, 16, device=device)).shape == (4, 3)
print(counts)


In [ ]:
cnn_train_ds, cnn_val_ds = split_dataset(TensorDataset(cnn_images, cnn_labels))
cnn_train_loader = DataLoader(cnn_train_ds, batch_size=32, shuffle=True)
cnn_val_loader = DataLoader(cnn_val_ds, batch_size=64, shuffle=False)
cnn_loss_fn = nn.CrossEntropyLoss()
cnn_optimizer = optim.AdamW(cnn_model.parameters(), lr=0.005)
for epoch in range(7):
    tr_loss, _ = run_multiclass_epoch(cnn_model, cnn_train_loader, cnn_loss_fn, cnn_optimizer)
    va_loss, va_acc = run_multiclass_epoch(cnn_model, cnn_val_loader, cnn_loss_fn)
print(f"CNN val loss={va_loss:.4f} accuracy={va_acc:.3f}")
assert va_acc > 0.85, "Synthetic pattern task should be learnable"


### Debugging exercise

A batch arrives as `(B,H,W,C)`. Diagnose the `Conv2d` channel error and write the fix. When is `x.permute(0,3,1,2)` correct, and why should you inspect shape instead of applying it blindly?


In [ ]:
broken_nhwc = torch.randn(8, 16, 16, 1)
fixed_nchw = broken_nhwc.permute(0, 3, 1, 2)
assert fixed_nchw.shape == (8, 1, 16, 16)
print("fixed:", fixed_nchw.shape)


**What would you try next?** Random shifts as augmentation, a linear pixel baseline, error inspection by pattern/offset, or a smaller CNN if latency matters.


### Before moving on

- [ ] I understand input and target shapes.
- [ ] I chose the loss and built the model myself.
- [ ] I wrote training and validation loops and calculated the metric.
- [ ] I can explain the architecture and why the loss is correct.
- [ ] I can describe at least two likely bugs and one sensible improvement.


## 7. Project 5 — Text Classification with Embeddings

Classify variable-length integer token sequences. Class-specific tokens come mostly from ranges `1–50`, `51–100`, or `101–150`; noise comes from the rest of a 500-token vocabulary.

### 🔥 Interview Useful for NLP

Embed `(B,T)` IDs into `(B,T,D)`, mask padding, and pool only real tokens.


In [ ]:
vocab_size_text, num_classes_text = 500, 3
def make_text_sample(label):
    length = random.randint(5, 30)
    class_lo = 1 + label * 50; class_hi = class_lo + 49
    n_signal = max(3, int(length * 0.7))
    tokens = [random.randint(class_lo, class_hi) for _ in range(n_signal)]
    tokens += [random.randint(151, vocab_size_text - 1) for _ in range(length - n_signal)]
    random.shuffle(tokens)
    return torch.tensor(tokens, dtype=torch.long)

random.seed(505)
text_sequences = [make_text_sample(i % 3) for i in range(1500)]
text_labels = [i % 3 for i in range(1500)]
order = list(range(1500)); random.shuffle(order)
text_sequences = [text_sequences[i] for i in order]
text_labels = [text_labels[i] for i in order]
print("length range:", min(map(len, text_sequences)), max(map(len, text_sequences)))


In [ ]:
# Your turn: implement TextDataset returning {"tokens": ..., "label": ...}.
TextDatasetDraft = None
pending("TextDataset", TextDatasetDraft)


In [ ]:
class TextDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences, self.labels = sequences, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {"tokens": self.sequences[index],
                "label": torch.tensor(self.labels[index], dtype=torch.long)}

text_dataset = TextDataset(text_sequences, text_labels)
sample = text_dataset[0]
assert set(sample) == {"tokens", "label"} and sample["tokens"].dtype == torch.long
print(sample["tokens"].shape, sample["label"])


In [ ]:
# Your turn: dynamically pad and return input_ids, attention_mask, labels.
collate_batch_draft = None
pending("collate_batch", collate_batch_draft)


In [ ]:
def collate_batch(batch):
    sequences = [item["tokens"] for item in batch]
    input_ids = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=0)
    attention_mask = (input_ids != 0).float()
    labels = torch.stack([item["label"] for item in batch])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

text_train_ds, text_val_ds = split_dataset(text_dataset)
text_train_loader = DataLoader(text_train_ds, batch_size=32, shuffle=True, collate_fn=collate_batch)
text_val_loader = DataLoader(text_val_ds, batch_size=64, shuffle=False, collate_fn=collate_batch)
batch = next(iter(text_train_loader))
assert batch["input_ids"].ndim == batch["attention_mask"].ndim == 2
assert batch["labels"].shape == (32,) and batch["labels"].dtype == torch.long
print({k: tuple(v.shape) for k, v in batch.items()})


### Masked mean pooling — reason through shapes

`Embedding`: `(B,T) → (B,T,64)`. Mask unsqueeze: `(B,T) → (B,T,1)`. Multiply, sum over `T`, divide by valid-token counts, producing `(B,64)`. Why must lengths be clamped?


In [ ]:
# Your turn: Embedding(500,64) → masked mean → Linear(64,32) → ReLU → 3.
text_model_draft = None
pending("text classifier", text_model_draft)


In [ ]:
class MeanPoolTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.classifier = nn.Sequential(nn.Linear(embed_dim, 32), nn.ReLU(), nn.Linear(32, num_classes))
    def forward(self, input_ids, attention_mask):
        embeddings = self.embedding(input_ids)              # (B,T,D)
        mask = attention_mask.unsqueeze(-1)                 # (B,T,1)
        summed = (embeddings * mask).sum(dim=1)              # (B,D)
        lengths = mask.sum(dim=1).clamp(min=1)               # (B,1)
        pooled = summed / lengths                            # (B,D)
        return self.classifier(pooled)                       # (B,C)

text_model = MeanPoolTextClassifier(vocab_size_text).to(device)
test_batch = next(iter(text_train_loader))
assert text_model(test_batch["input_ids"].to(device), test_batch["attention_mask"].to(device)).shape == (32, 3)
print("✅ text model ready")


In [ ]:
def run_text_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None; model.train(training)
    total_loss = correct = total = 0
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for batch in loader:
            ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            if training: optimizer.zero_grad()
            logits = model(ids, mask); loss = loss_fn(logits, labels)
            if training: loss.backward(); optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item(); total += labels.numel()
    return total_loss / len(loader.dataset), correct / total

text_loss_fn = nn.CrossEntropyLoss()
text_optimizer = optim.AdamW(text_model.parameters(), lr=0.005)
text_history = []
for epoch in range(7):
    train_result = run_text_epoch(text_model, text_train_loader, text_loss_fn, text_optimizer)
    val_result = run_text_epoch(text_model, text_val_loader, text_loss_fn)
    text_history.append((train_result, val_result))
print(f"text val loss={val_result[0]:.4f} accuracy={val_result[1]:.3f}")
assert val_result[1] > 0.85


### Text interview discussion

- Why embeddings instead of one-hot vectors? Why does `Embedding` require integer IDs?
- Why dynamic padding and masking? What bias appears if padding is averaged?
- Why mean pooling? What order information does it discard?
- Where would a tokenizer normally fit? How could a Transformer improve the baseline?
- **What would you try next?** Inspect length/class slices, try max pooling or a small sequence model, and improve tokenization only when data warrants it.


<details><summary>Concise answers</summary>

Embedding performs an efficient learned lookup instead of multiplying huge sparse one-hots. Padding makes rectangular batches; masking prevents fake tokens from diluting shorter examples. Mean pooling is cheap and length-stable but discards order. A tokenizer maps raw text to IDs before this dataset; a Transformer can model order and context.

</details>


### Before moving on

- [ ] I understand input and target shapes.
- [ ] I chose the loss and built the model myself.
- [ ] I wrote training and validation loops and calculated the metric.
- [ ] I can explain the architecture and why the loss is correct.
- [ ] I can describe at least two likely bugs and one sensible improvement.


## 8. Optional Project 6 — Tiny Transformer Language Model

### 🔴 Hard / Optional

Predict the next token in sequences that mostly increment modulo a 128-token vocabulary. This demonstrates mechanics, not production LLM generation.


In [ ]:
vocab_size_lm, hidden_size_lm, num_heads_lm, num_layers_lm = 128, 64, 4, 2
torch.manual_seed(606)
starts = torch.randint(0, vocab_size_lm, (640, 1))
steps = torch.arange(17).unsqueeze(0)
lm_full = (starts + steps) % vocab_size_lm
lm_inputs, lm_targets = lm_full[:, :-1], lm_full[:, 1:]
lm_loader = DataLoader(TensorDataset(lm_inputs, lm_targets), batch_size=64, shuffle=True)
assert lm_inputs.shape == lm_targets.shape == (640, 16)


In [ ]:
# Your turn: token + position embeddings, causal Transformer, norm, vocab head.
tiny_lm_draft = None
pending("tiny language model", tiny_lm_draft)


In [ ]:
class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size=128, hidden_size=64, heads=4, layers=2, max_len=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_size)
        self.position_embedding = nn.Embedding(max_len, hidden_size)
        layer = nn.TransformerEncoderLayer(hidden_size, heads, 128, dropout=0.0, batch_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers)
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size)
    def forward(self, token_ids):
        b, t = token_ids.shape
        positions = torch.arange(t, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)[None, :, :]
        causal_mask = nn.Transformer.generate_square_subsequent_mask(t, device=token_ids.device)
        x = self.blocks(x, mask=causal_mask, is_causal=True)
        return self.head(self.norm(x))

tiny_lm = TinyLanguageModel().to(device)
lm_loss_fn = nn.CrossEntropyLoss(); lm_optimizer = optim.AdamW(tiny_lm.parameters(), lr=0.003)
for epoch in range(5):
    tiny_lm.train(); running = 0.0
    for ids, targets in lm_loader:
        ids, targets = ids.to(device), targets.to(device)
        lm_optimizer.zero_grad(); logits = tiny_lm(ids)
        loss = lm_loss_fn(logits.reshape(-1, vocab_size_lm), targets.reshape(-1))
        loss.backward(); lm_optimizer.step(); running += loss.item() * ids.size(0)
print(f"LM loss={running / len(lm_loader.dataset):.4f}")
assert tiny_lm(torch.ones(2, 5, dtype=torch.long, device=device)).shape == (2, 5, 128)


In [ ]:
tiny_lm.eval()
generated = torch.tensor([[1, 2, 3]], dtype=torch.long, device=device)
with torch.inference_mode():
    for _ in range(8):
        next_logits = tiny_lm(generated)[:, -1, :]
        next_token = next_logits.argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
print("greedy generation:", generated.cpu().tolist())


## 9. Compare the Projects

### 🔥 Memorize This Table

| Project | Output | Target | Loss | Prediction |
|---|---|---|---|---|
| Regression | `(B,1)` | float `(B,1)` | MSE | raw value |
| Binary | `(B,1)` logits | float `(B,1)` | BCEWithLogits | sigmoid + threshold |
| Multiclass | `(B,C)` logits | long `(B,)` | CrossEntropy | argmax |
| CNN multiclass | `(B,C)` logits | long `(B,)` | CrossEntropy | argmax |
| Text multiclass | `(B,C)` logits | long `(B,)` | CrossEntropy | argmax |
| Language model | `(B,T,V)` logits | long `(B,T)` | CrossEntropy | next-token selection |


## 10. Architecture Decision Questions


**1. 40 numeric features and five exclusive classes: what would you choose first, and why?**


<details><summary>Answer</summary>

Start with an MLP (and compare a linear baseline); output `(B,5)`.

</details>


**2. Token IDs `(B,T)` need dense vectors: what would you choose first, and why?**


<details><summary>Answer</summary>

Use `nn.Embedding`; it maps integer IDs to `(B,T,D)`.

</details>


**3. Images arrive as `(B,3,224,224)`: what would you choose first, and why?**


<details><summary>Answer</summary>

A CNN/`Conv2d` is a natural baseline, though other image architectures are valid.

</details>


**4. Long text needs order-sensitive contextual processing: what would you choose first, and why?**


<details><summary>Answer</summary>

A Transformer is a strong option; simpler recurrent/convolutional baselines may also work.

</details>


**5. One continuous house price: what would you choose first, and why?**


<details><summary>Answer</summary>

One raw continuous output and a regression loss such as MSE or L1.

</details>


**6. Mutually exclusive 12-class target: what would you choose first, and why?**


<details><summary>Answer</summary>

Twelve logits with CrossEntropy and long class indices.

</details>


**7. Eight independent tags can co-occur: what would you choose first, and why?**


<details><summary>Answer</summary>

Eight logits, float multi-hot targets, BCEWithLogits per label.

</details>


**8. Binary fraud score: what would you choose first, and why?**


<details><summary>Answer</summary>

One logit is the simplest representation; threshold sigmoid probabilities based on costs.

</details>


**9. Variable-length text batches: what would you choose first, and why?**


<details><summary>Answer</summary>

Use padding plus a mask; dynamic padding reduces wasted computation.

</details>


**10. Very small tabular dataset: what would you choose first, and why?**


<details><summary>Answer</summary>

Start with a linear/small MLP baseline and strong validation; complexity can overfit.

</details>


**11. Spatial location should matter locally: what would you choose first, and why?**


<details><summary>Answer</summary>

Convolutions encode locality and weight sharing.

</details>


**12. Sequence order should not matter: what would you choose first, and why?**


<details><summary>Answer</summary>

Masked mean/sum pooling can be an efficient baseline.

</details>


**13. Predict next token at every position: what would you choose first, and why?**


<details><summary>Answer</summary>

Return `(B,T,V)` and compare each position with a long target token ID.

</details>


**14. Targets have large outliers: what would you choose first, and why?**


<details><summary>Answer</summary>

Consider L1/Huber and investigate data quality rather than choosing by habit.

</details>


**15. Positive class is rare: what would you choose first, and why?**


<details><summary>Answer</summary>

Keep a suitable binary model; change sampling/loss/threshold/metrics based on costs.

</details>


**16. Image resolution changes: what would you choose first, and why?**


<details><summary>Answer</summary>

Use adaptive pooling or infer flatten dimensions instead of hard-coding spatial size.

</details>


**17. Need calibrated binary probabilities: what would you choose first, and why?**


<details><summary>Answer</summary>

Train logits correctly, then assess calibration and possibly calibrate on held-out data.

</details>


**18. Class IDs are strings: what would you choose first, and why?**


<details><summary>Answer</summary>

Map them to contiguous integer indices before CrossEntropy.

</details>


**19. Need a fast text baseline: what would you choose first, and why?**


<details><summary>Answer</summary>

Embedding + masked mean pooling + linear classifier.

</details>


**20. Structured sequence has local and global dependencies: what would you choose first, and why?**


<details><summary>Answer</summary>

Start with a sequence model chosen for length/latency; CNN, RNN, or Transformer may fit.

</details>


## 11. Project Debugging Scenarios


**Scenario 1.** Regression prediction `(B,1)`, target `(B,)`; MSE warns. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Match shapes, usually `target = target.unsqueeze(1)`. Broadcasting could create `(B,B)` errors.

</details>


**Scenario 2.** Binary model has Sigmoid and uses BCEWithLogitsLoss. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Remove Sigmoid from the model; the loss already combines it stably.

</details>


**Scenario 3.** Five-class logits, but labels are float. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Use long class indices shaped `(B,)` for CrossEntropy.

</details>


**Scenario 4.** CNN Linear expects 1024 but flatten gives 256. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Trace shapes after each layer; set `in_features=256` or use adaptive pooling/lazy initialization carefully.

</details>


**Scenario 5.** Text pooling averages padded positions. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Multiply embeddings by the attention mask and divide by valid-token counts.

</details>


**Scenario 6.** Loss is flat from the first step. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Check data/labels, learning rate, gradients, optimizer parameters, `zero_grad/backward/step`, and whether inputs carry signal.

</details>


**Scenario 7.** Every parameter gradient is `None`. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Look for `no_grad/inference_mode`, detached tensors, non-differentiable conversions, or loss disconnected from the model.

</details>


**Scenario 8.** Gradients exist but parameters never change. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Check `optimizer.step()`, optimizer parameter references, zero learning rate, and compare snapshots before/after step.

</details>


**Scenario 9.** CUDA tensors meet a CPU model. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Move model and every input/target tensor to the same device.

</details>


**Scenario 10.** Validation accuracy changes wildly each call. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Call `eval()` to disable dropout/update behavior, keep a fixed split, and avoid validation augmentation/shuffling assumptions.

</details>


**Scenario 11.** Validation changes model parameters. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Use `eval()` plus `inference_mode()` and never call optimizer steps in validation; compare parameter snapshots.

</details>


**Scenario 12.** CrossEntropy receives softmax probabilities. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Pass raw logits; CrossEntropy performs log-softmax internally.

</details>


**Scenario 13.** Binary target is long and shaped `(B,)`. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Convert to float and match one-logit shape `(B,1)`.

</details>


**Scenario 14.** Accuracy is high but rare-class recall is zero. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Inspect confusion counts and precision/recall; tune sampling, weighting, or threshold around business costs.

</details>


**Scenario 15.** Training loss is NaN. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Check finite data/logits/loss, learning rate, divisions/logs, exploding gradients, and the first failing batch.

</details>


**Scenario 16.** DataLoader repeats identical order each epoch unexpectedly. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Confirm `shuffle=True` for training and that a sampler is not overriding it.

</details>


**Scenario 17.** Embedding throws an index-out-of-range error. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Inspect min/max token IDs; reserve valid padding/unknown IDs below `num_embeddings`.

</details>


**Scenario 18.** All text examples pool to nearly zero. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Inspect mask values/dtype and lengths; ensure real tokens are marked 1 and padding 0.

</details>


**Scenario 19.** CNN receives NHWC. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

When confirmed, permute to NCHW with `x.permute(0,3,1,2)`.

</details>


**Scenario 20.** OOM occurs after several validation iterations. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Use inference mode and avoid retaining logits/loss graphs; move only needed summaries to CPU.

</details>


**Scenario 21.** Top-k code uses `dim=0`. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

For `(B,C)` logits choose classes along `dim=1`.

</details>


**Scenario 22.** Confusion matrix looks transposed. What do you inspect and fix?


<details><summary>Reveal diagnosis</summary>

Document convention and increment `[actual, predicted]` consistently.

</details>


## 12. Interview Walkthrough Practice

For every required project, answer: (1) problem, (2) one input, (3) one target, (4) model choice, (5) loss, (6) optimizer, (7) metric, (8) checked bugs, (9) next improvement. Then give a conversational 60-second explanation.


### Linear regression: explain this project to an interviewer in 60 seconds


<details><summary>Example answer</summary>

I predicted one continuous value from two numeric features. The MLP maps `(B,2)` to `(B,1)` with no output activation. I used MSE with Adam, tracked validation MSE and MAE, and checked shape broadcasting, gradients, parameter updates, and device placement. The synthetic relationship is a sanity check; next I would compare the MLP with a plain linear model and inspect residuals.

</details>


### Binary classification: explain this project to an interviewer in 60 seconds


<details><summary>Example answer</summary>

I classified ten-feature samples with one raw logit. Float labels have the same `(B,1)` shape, so BCEWithLogitsLoss is numerically stable. At inference I apply sigmoid and a threshold. Besides accuracy I computed precision, recall, and confusion counts because the threshold controls real decision costs.

</details>


### Multiclass classification: explain this project to an interviewer in 60 seconds


<details><summary>Example answer</summary>

I predicted one of five classes with five logits and long class-index targets. The baseline is a small MLP trained with CrossEntropyLoss and AdamW. I tracked loss and accuracy, built an actual-row/predicted-column confusion matrix, and measured top-3 accuracy. I would inspect confused classes before adding capacity.

</details>


### CNN image classification: explain this project to an interviewer in 60 seconds


<details><summary>Example answer</summary>

I generated three learnable image patterns in NCHW format and used two convolution/pooling blocks plus a small classifier. CrossEntropyLoss matches the three logits. I calculated feature-map shapes and parameter counts before training, then checked channel order and the flatten dimension—the most common implementation bugs here.

</details>


### Text classification: explain this project to an interviewer in 60 seconds


<details><summary>Example answer</summary>

I represented variable-length samples as token IDs, dynamically padded them, and created an attention mask. A learned embedding plus masked mean pooling gives a fast baseline before the three-class head. CrossEntropyLoss fits long class labels. I verified padding contributes neither to the sum nor denominator; a Transformer is a possible next step if order and context matter.

</details>


### Training history interpretation

- **Case A:** train loss ↓, validation loss ↓. What does this support?
- **Case B:** train loss ↓, validation loss ↑ after epoch 8. What is likely happening?
- **Case C:** both remain high. What should you inspect before changing architecture?


<details><summary>Interpretation</summary>

A: the model is learning patterns that generalize so far. B: likely overfitting; confirm across runs and metrics. C: possible underfitting or an optimization/data bug—inspect a batch, label signal, loss/target contract, gradients, updates, learning rate, and whether a tiny subset can be overfit.

</details>


## 13. Project-Oriented Interview Questions


**1. How do you decide output dimension?**


<details><summary>Short Interview Answer</summary>

Derive it from one target and the prediction contract.

</details>


<details><summary>Detailed Explanation</summary>

Regression needs the continuous target shape; binary commonly uses one logit; exclusive C-class classification uses C logits; language modeling uses one V-logit vector per position.

</details>


**2. How do you choose loss?**


<details><summary>Short Interview Answer</summary>

Match it to output semantics and target representation.

</details>


<details><summary>Detailed Explanation</summary>

Check regression vs classification, exclusive vs independent labels, logits vs probabilities, target shape/dtype, and error costs.

</details>


**3. Why one raw logit for binary classification?**


<details><summary>Short Interview Answer</summary>

It is sufficient and pairs stably with BCEWithLogitsLoss.

</details>


<details><summary>Detailed Explanation</summary>

Sigmoid maps the logit to a probability at inference; the fused loss avoids numerical instability.

</details>


**4. Why one logit per multiclass class?**


<details><summary>Short Interview Answer</summary>

CrossEntropy compares competing class scores.

</details>


<details><summary>Detailed Explanation</summary>

The correct class index selects the target logit while normalization is handled internally.

</details>


**5. How do you choose target dtype?**


<details><summary>Short Interview Answer</summary>

Follow the loss contract.

</details>


<details><summary>Detailed Explanation</summary>

MSE/BCE generally take floats; CrossEntropy with class indices takes long.

</details>


**6. Good first tabular architecture?**


<details><summary>Short Interview Answer</summary>

Linear model or small MLP.

</details>


<details><summary>Detailed Explanation</summary>

Start with the smallest baseline that can reveal data and pipeline issues; compare complexity against validation evidence.

</details>


**7. When use a CNN?**


<details><summary>Short Interview Answer</summary>

When local spatial structure and weight sharing are useful.

</details>


<details><summary>Detailed Explanation</summary>

Images are common, but convolutions also work for other grid/local sequence problems.

</details>


**8. When use an embedding?**


<details><summary>Short Interview Answer</summary>

For categorical IDs that need learned dense vectors.

</details>


<details><summary>Detailed Explanation</summary>

It is an efficient lookup table, especially for tokens or large-cardinality categories.

</details>


**9. Why dynamically pad text?**


<details><summary>Short Interview Answer</summary>

It pads only to the longest item in each batch.

</details>


<details><summary>Detailed Explanation</summary>

This usually reduces wasted memory/compute versus padding every sample to a global maximum.

</details>


**10. Why mask padding during pooling?**


<details><summary>Short Interview Answer</summary>

Padding is not content.

</details>


<details><summary>Detailed Explanation</summary>

Including it dilutes short sequences and can create length-dependent bias.

</details>


**11. Why separate train and validation data?**


<details><summary>Short Interview Answer</summary>

To estimate generalization on examples not used for updates.

</details>


<details><summary>Detailed Explanation</summary>

Using training data for model selection produces optimistic estimates.

</details>


**12. Why shuffle training?**


<details><summary>Short Interview Answer</summary>

It reduces order bias in minibatch updates.

</details>


<details><summary>Detailed Explanation</summary>

It improves stochastic mixing; special sequential tasks may require different sampling.

</details>


**13. Why needn't validation shuffle?**


<details><summary>Short Interview Answer</summary>

No gradient updates depend on order.

</details>


<details><summary>Detailed Explanation</summary>

Fixed order improves reproducibility and makes example-level debugging easier.

</details>


**14. How tell if a model is learning?**


<details><summary>Short Interview Answer</summary>

Track train/validation loss and task metrics against a baseline.

</details>


<details><summary>Detailed Explanation</summary>

Also verify gradients, parameter changes, and ability to overfit a tiny subset.

</details>


**15. How detect overfitting?**


<details><summary>Short Interview Answer</summary>

Training improves while validation degrades or stalls.

</details>


<details><summary>Detailed Explanation</summary>

Use trends across epochs/runs and relevant metrics, not one noisy point.

</details>


**16. How debug flat loss?**


<details><summary>Short Interview Answer</summary>

Check data, contract, gradients, updates, and learning rate.

</details>


<details><summary>Detailed Explanation</summary>

Inspect the first batch end-to-end and try to overfit a tiny batch before redesigning the network.

</details>


**17. How verify gradients?**


<details><summary>Short Interview Answer</summary>

Inspect `p.grad` after backward.

</details>


<details><summary>Detailed Explanation</summary>

Check for `None`, non-finite values, and unexpectedly zero norms before the optimizer step.

</details>


**18. How verify parameter updates?**


<details><summary>Short Interview Answer</summary>

Snapshot parameters before and compare after `step()`.

</details>


<details><summary>Detailed Explanation</summary>

Use detached clones; comparing references to the same storage is misleading.

</details>


**19. Why `eval()` for validation?**


<details><summary>Short Interview Answer</summary>

It switches modules such as dropout and batch norm to inference behavior.

</details>


<details><summary>Detailed Explanation</summary>

It does not disable gradients by itself, so pair it with inference mode.

</details>


**20. Why disable Autograd in validation?**


<details><summary>Short Interview Answer</summary>

It saves memory and compute and prevents graph retention.

</details>


<details><summary>Detailed Explanation</summary>

`torch.inference_mode()` is ideal when no gradients are needed.

</details>


**21. How handle class imbalance?**


<details><summary>Short Interview Answer</summary>

Choose cost-aware metrics first, then sampling, weighting, or thresholds.

</details>


<details><summary>Detailed Explanation</summary>

Evaluate per-class behavior and avoid treating accuracy as sufficient.

</details>


**22. How handle GPU OOM?**


<details><summary>Short Interview Answer</summary>

Reduce batch/sequence/image size and inspect retained graphs.

</details>


<details><summary>Detailed Explanation</summary>

Gradient accumulation, mixed precision, checkpointing, and smaller models can help when appropriate.

</details>


**23. How speed training?**


<details><summary>Short Interview Answer</summary>

Profile the bottleneck before optimizing.

</details>


<details><summary>Detailed Explanation</summary>

Tune input pipeline, vectorize, use suitable precision/device, and avoid unnecessary transfers or logging.

</details>


**24. What log during training?**


<details><summary>Short Interview Answer</summary>

Loss, relevant metrics, learning rate, timing, and validation trends.

</details>


<details><summary>Detailed Explanation</summary>

For debugging also log shapes, gradient norms, class counts, and representative errors.

</details>


**25. How explain architecture?**


<details><summary>Short Interview Answer</summary>

Connect each layer to input structure and output contract.

</details>


<details><summary>Detailed Explanation</summary>

Mention baseline rationale, shapes, capacity, regularization, constraints, and alternatives.

</details>


**26. When is complexity justified?**


<details><summary>Short Interview Answer</summary>

When a simpler verified baseline misses patterns and validation evidence supports it.

</details>


<details><summary>Detailed Explanation</summary>

Consider data size, latency, memory, interpretability, and maintenance.

</details>


**27. Why start with a baseline?**


<details><summary>Short Interview Answer</summary>

It establishes a trustworthy reference and validates the pipeline.

</details>


<details><summary>Detailed Explanation</summary>

Without it, gains and failures are hard to interpret.

</details>


**28. What changes from regression to classification?**


<details><summary>Short Interview Answer</summary>

Output semantics, target representation, loss, and metrics.

</details>


<details><summary>Detailed Explanation</summary>

The data pipeline/model trunk may remain similar, but the prediction head and evaluation contract change.

</details>


**29. Binary versus multiclass?**


<details><summary>Short Interview Answer</summary>

One float logit/BCE versus C logits/long class index/CrossEntropy.

</details>


<details><summary>Detailed Explanation</summary>

Multilabel is different again: C independent logits with float multi-hot targets and BCE.

</details>


**30. How productionize at a high level?**


<details><summary>Short Interview Answer</summary>

Freeze preprocessing and model contracts, package inference, monitor, and test.

</details>


<details><summary>Detailed Explanation</summary>

Version artifacts, validate inputs, benchmark latency, monitor drift/quality, secure deployment, and design rollback/retraining.

</details>


## 14. Final From-Scratch Challenge

No architecture hints. Each sample has 30 features and belongs to one of six categories. Build the complete baseline independently.


In [ ]:
torch.manual_seed(707)
X_final = torch.randn(1800, 30)
final_rule = torch.randn(30, 6)
y_final = (X_final @ final_rule + 0.25 * torch.randn(1800, 6)).argmax(dim=1)
assert X_final.shape == (1800, 30) and y_final.shape == (1800,)
print(X_final.shape, y_final.shape, y_final.dtype)


Decide and implement: (1) split, (2) Dataset/DataLoader, (3) model, (4) output dimension, (5) loss, (6) optimizer, (7) train loop, (8) validation loop, (9) accuracy, (10) device handling.

Required evidence: `Dataset/DataLoader`, `nn.Module` or `Sequential`, correct loss/dtype, reasonable optimizer, `train()`, `eval()`, inference mode, device consistency, and a correct metric.


In [ ]:
# Your complete solution belongs here.
final_model_draft = None
final_loss_draft = None
final_history_draft = None
pending("final from-scratch challenge", final_model_draft, final_loss_draft, final_history_draft)


In [ ]:
final_train_ds, final_val_ds = split_dataset(TensorDataset(X_final, y_final))
final_train_loader = DataLoader(final_train_ds, batch_size=32, shuffle=True)
final_val_loader = DataLoader(final_val_ds, batch_size=64, shuffle=False)
final_model = nn.Sequential(nn.Linear(30, 64), nn.ReLU(), nn.Linear(64, 6)).to(device)
final_loss_fn = nn.CrossEntropyLoss()
final_optimizer = optim.AdamW(final_model.parameters(), lr=0.01)
final_before = [p.detach().cpu().clone() for p in final_model.parameters()]
for epoch in range(10):
    final_train = run_multiclass_epoch(final_model, final_train_loader, final_loss_fn, final_optimizer)
    final_val = run_multiclass_epoch(final_model, final_val_loader, final_loss_fn)
final_after = [p.detach().cpu().clone() for p in final_model.parameters()]
xb, yb = next(iter(final_val_loader))
assert final_model(xb.to(device)).shape == (xb.size(0), 6)
assert yb.dtype == torch.long and isinstance(final_loss_fn, nn.CrossEntropyLoss)
assert any(not torch.equal(a, b) for a, b in zip(final_before, final_after))
assert 0 <= final_val[1] <= 1
print(f"final validation loss={final_val[0]:.4f} accuracy={final_val[1]:.3f}")


### Follow-up

How would the solution change if this were binary? How would it change for a continuous target?


<details><summary>Answer</summary>

Binary: one raw logit, float target `(B,1)`, BCEWithLogitsLoss, sigmoid + threshold. Continuous: task-shaped raw regression output, float target, MSE/L1 (or another justified regression loss), and MAE/MSE/RMSE rather than classification accuracy.

</details>


## 15. Final Project Comparison Challenge

For each scenario, choose **model family, output shape, loss, target dtype, and metric** before revealing the reference. Alternatives are valid when justified.


**1. House price**


<details><summary>Reference choice</summary>

Model: MLP/linear; output: `(B,1)`; loss: MSE/L1; target: float; metric: MAE/RMSE.

</details>


**2. Fraud yes/no**


<details><summary>Reference choice</summary>

Model: MLP; output: `(B,1)`; loss: BCEWithLogits; target: float; metric: precision/recall/PR-AUC.

</details>


**3. 100-class image**


<details><summary>Reference choice</summary>

Model: CNN; output: `(B,100)`; loss: CrossEntropy; target: long; metric: top-1/top-k.

</details>


**4. Independent image tags**


<details><summary>Reference choice</summary>

Model: CNN; output: `(B,C)`; loss: BCEWithLogits; target: float multi-hot; metric: per-label F1/mAP.

</details>


**5. Three-way sentiment**


<details><summary>Reference choice</summary>

Model: embedding/Transformer; output: `(B,3)`; loss: CrossEntropy; target: long; metric: accuracy/macro-F1.

</details>


**6. Next-token LM**


<details><summary>Reference choice</summary>

Model: Transformer; output: `(B,T,V)`; loss: CrossEntropy; target: long; metric: loss/perplexity.

</details>


**7. Tabular six-class**


<details><summary>Reference choice</summary>

Model: MLP; output: `(B,6)`; loss: CrossEntropy; target: long; metric: accuracy/macro-F1.

</details>


**8. Customer churn**


<details><summary>Reference choice</summary>

Model: MLP; output: `(B,1)`; loss: BCEWithLogits; target: float; metric: recall/ROC-AUC.

</details>


**9. Age prediction**


<details><summary>Reference choice</summary>

Model: MLP/CNN; output: `(B,1)`; loss: L1/MSE; target: float; metric: MAE.

</details>


**10. Five simultaneous attributes**


<details><summary>Reference choice</summary>

Model: task-appropriate trunk; output: `(B,5)`; loss: BCEWithLogits; target: float multi-hot; metric: macro/micro-F1.

</details>


**11. Ten-class audio clip**


<details><summary>Reference choice</summary>

Model: audio CNN; output: `(B,10)`; loss: CrossEntropy; target: long; metric: accuracy.

</details>


**12. Binary token classification**


<details><summary>Reference choice</summary>

Model: sequence model; output: `(B,T,1)`; loss: masked BCEWithLogits; target: float; metric: token F1.

</details>


**13. NER with K tags**


<details><summary>Reference choice</summary>

Model: sequence model; output: `(B,T,K)`; loss: masked CrossEntropy; target: long; metric: entity F1.

</details>


**14. Two continuous targets**


<details><summary>Reference choice</summary>

Model: MLP; output: `(B,2)`; loss: MSE/L1; target: float; metric: per-target MAE.

</details>


**15. Rank one of 20 candidates**


<details><summary>Reference choice</summary>

Model: ranking/scoring model; output: task-dependent; loss: ranking/CE; target: task-dependent; metric: MRR/Recall@k.

</details>


## 16. Project Cheat Sheet

### Regression
Output: `(B,1)` or task-specific continuous shape · target: float · loss: MSE/L1 · prediction: raw output.

### Binary classification
Output: `(B,1)` raw logits · target: float `(B,1)` · loss: BCEWithLogits · prediction: sigmoid → threshold.

### Multiclass classification
Output: `(B,C)` raw logits · target: `(B,)` long · loss: CrossEntropy · prediction: `argmax(dim=1)`.

### Multilabel classification
Output: `(B,C)` raw logits · target: `(B,C)` float · loss: BCEWithLogits · prediction: sigmoid → threshold per class.

### Language modeling
Output: `(B,T,V)` · target: `(B,T)` long · loss: CrossEntropy over flattened token positions.


## 17. Critical Mental Model

```text
New ML problem
      ↓
What is one input?
      ↓
What is one target?
      ↓
What should model output?
      ↓
What loss matches that output/target?
      ↓
What architecture is a reasonable baseline?
      ↓
Build data pipeline → Train → Validate → Debug → Improve
```

### 🔥 This is more important than memorizing architectures.


## PyTorch Project Readiness

- [ ] I can build regression, binary, and multiclass models from scratch.
- [ ] I can build a basic CNN and embedding-based text classifier.
- [ ] I can choose loss, output shape, target shape, and dtype without hints.
- [ ] I can write Dataset/DataLoader, training, validation, and device-aware code.
- [ ] I can calculate appropriate metrics and debug common failures.
- [ ] I can explain architecture choices and a project clearly in an interview.
- [ ] I can take a new ML prompt and build a verified baseline without copying a template.
